## Exercise 1 Function Approximation

#### (*) 1.1 What is the difference between using a linear function for feature engineering and function approximation?

The difference is mainly what is being approximated and where the linear function is applied.

In feature engineering, the original state is transformed into a feature vector. And the goal is to represent the state in a simpler or more meaningful way before learning new values. Instead of storing a value for each state, values are stored for features. Essentially we  change the representation of the state, not necessarily how the value is computed.

In function approximation, we directly approximate the value function using a parameterized function. So we essentially learn a function that maps the features from feature engineering to values.
With feature engineering we create the inputs, and then function approximation learns a function that maps those features to value estimates.

#### 1.2 Write out the semi-gradient for learning a function approximation when performing prediction with a TD(n) target.
For value prediction with function approximation $\hat{v}(s,\mathbf{w})$, the $n$-step TD target is

$$
G_{t:t+n}
=
R_{t+1} + \gamma R_{t+2} + \cdots + \gamma^{n-1} R_{t+n} + \gamma^n \hat{v}(S_{t+n}, \mathbf{w})
$$

assuming $S_{t+n}$ is nonterminal.

The semi-gradient for learning from this TD($n$) target is

$$
\left(G_{t:t+n} - \hat{v}(S_t,\mathbf{w})\right)\nabla_{\mathbf{w}} \hat{v}(S_t,\mathbf{w}).
$$

That is, (bonus stuff, not actually needed but for clarification and explanation I wrote it out more)

$$
\left(
R_{t+1}
+ \gamma R_{t+2}
+ \cdots
+ \gamma^{n-1} R_{t+n}
+ \gamma^n \hat{v}(S_{t+n},\mathbf{w})
- \hat{v}(S_t,\mathbf{w})
\right)
\nabla_{\mathbf{w}} \hat{v}(S_t,\mathbf{w}).
$$

So the corresponding semi-gradient update is

$$
\mathbf{w}_{t+1}
=
\mathbf{w}_t
+
\alpha
\left(
G_{t:t+n} - \hat{v}(S_t,\mathbf{w}_t)
\right)
\nabla_{\mathbf{w}} \hat{v}(S_t,\mathbf{w}_t).
$$

It is called a semi-gradient because the TD target $G_{t:t+n}$ also depends on $\mathbf{w}$ through the bootstrapped term $\hat{v}(S_{t+n},\mathbf{w})$, but when taking the gradient we ignore that dependence and differentiate only $\hat{v}(S_t,\mathbf{w})$.

#### (*) 1.3 Consider the policy improvement theorem that we discussed in the context of dynamic programming. Why do the guarantees of this theorem fail to apply in case of function approximation? What step of the proof does not hold anymore?

The guarantees of the policy improvement theorem fail when using function approximation because the theorem assumes that the value or action-value function of the current policy is known and exact.

In dynamic programming after policy evaluation we have the true action-value function $q_\pi(s,a)$.  
The improved policy is defined as

$$
\pi'(s) = \arg\max_a q_\pi(s,a)
$$

The proof of the policy improvement theorem relies on the inequality

$$
v_\pi(s) = \sum_a \pi(a|s) q_\pi(s,a) \le \max_a q_\pi(s,a) = q_\pi(s,\pi'(s))
$$

which guarantees that

$$
v_{\pi'}(s) \ge v_\pi(s)
$$

for all states $s$. Because of this the new policy is guaranteed to be at least as good as the previous one.

But in function approximation, we dont know the true value function. Instead we learn an approximation such as

$$
\hat{q}_\pi(s,a; w)
$$

or

$$
\hat{v}_\pi(s; w)
$$

The greedy improvement step then becomes

$$
\pi'(s) = \arg\max_a \hat{q}_\pi(s,a; w)
$$

Since $\hat{q}_\pi(s,a; w)$ is only an approximation, it may contain errors and may not maintain the correct ranking of actions. As a result

$$
\max_a \hat{q}_\pi(s,a; w)
$$

does not necessarily correspond to the action with the highest true value $q_\pi(s,a)$.

Therefore the key step in the proof

$$
v_\pi(s) \le \max_a q_\pi(s,a),
$$

no longer guarantees improvement when using approximated values. The greedy policy we got from the approximation may actually reduce the true value of the policy. Becuase of this the guarantees of the policy improvement theorem do not hold with function approximation.

#### (*) 1.4 Suppose you have been training the Mars Rover 2.0 robot using the tabularTD(λ) algorithm and relying on off-policy data gathered by the Mars Rover 1.0. A colleague suggests that to increase the performance of the Mars Rover 2.0 you should use a neural network to approximate its action-value function. What sort of challenge would you expect to face if you were to follow the suggestion of your colleague?

If we replace the tabular representation with a neural network to approximate the action-value function, we could face convergence and stability issues.

In the tabular TD($\lambda$) setting, the algorithm converges under suitable conditions because the value function is represented exactly for each state–action pair (each cell in the table). But if we were to use a neural network we introduce function approximation, which means we only learn an approximate action-value function $\hat{q}(s,a; w)$ instead of the true value function.

## Exercise 2 Model-free and Model-based learning

#### 2.1 What would be vπ(uni) and vπ(home) computed using MC?

For Monte Carlo prediction, we compute the value of a state as the average of the observed returns following visits to that state.

The state \(uni\) appears in all 8 trajectories.

The returns from \(uni\) are:

- $\tau_1: 1$
- $\tau_2: 1$
- $\tau_3: 0$
- $\tau_4: 1$
- $\tau_5: 1$
- $\tau_6: 1$
- $\tau_7: 1$
- $\tau_8: 0$

So,

$$
v_\pi(uni) = \frac{1+1+0+1+1+1+1+0}{8} = \frac{6}{8} = \frac{3}{4}
$$


The state $(home)$ appears only in $\tau_8$:

$$
\tau_8 = \{(home, travel, 0), (uni, take exam, 0)\}
$$

Starting from $(home)$, the return is

$$
G = 0 + \gamma \cdot 0 = 0
$$

Therefore,

$$
v_\pi(home) = 0
$$

$$
v_\pi(uni) = \frac{3}{4}, \qquad v_\pi(home) = 0
$$

#### 2.2 What would be vπ(uni) and vπ(home) computed using TD?

For TD prediction we use the TD(0) update rule

$$
v(s) \leftarrow v(s) + \alpha \left(r + \gamma v(s') - v(s)\right)
$$

Since episodes terminate after ($take exam$), the value of the terminal state is

$$
v(\text{terminal}) = 0
$$

In all trajectories where the agent starts in $uni$, the transition is

$(uni, take\ exam, r)$

and the episode terminates immediately. Therefore the TD target is

$$
r + \gamma v(\text{terminal}) = r
$$

Thus TD will converge to the expected reward when taking the exam in $uni$.

From the data we observe the rewards

$1,1,0,1,1,1,1,0$

Therefore

$$
v_\pi(uni) = \frac{6}{8} = \frac{3}{4}
$$


From the data we observe one transition from $home$:

$(home, travel, 0) \rightarrow uni$

The TD target is therefore

$$
0 + \gamma v_\pi(uni)
$$

Substituting the value of $v_\pi(uni)$ gives

$$
v_\pi(home) = \gamma \cdot \frac{3}{4}
$$


$$
v_\pi(uni) = \frac{3}{4}
$$

$$
v_\pi(home) = \frac{3}{4}\gamma
$$

#### (*) 2.3 Are the state values computed by MC and TD identical? If not, can you explain why not?

The state values computed by MC and TD are not identical.

Using MC:

$$
v_\pi(uni) = \frac{3}{4}, \qquad v_\pi(home) = 0
$$

Using TD:
$$
v_\pi(uni) = \frac{3}{4}, \qquad v_\pi(home) = \frac{3}{4}\gamma
$$

The reason for them not being identical is how they are being estimated by the different methods. MC estimates the value of a state using the full return from the episode. MC therefore waits until the episode terminates and then uses the actual observed return as the learning target. TD updates the value using a bootstrapped target based on the next state value estimate. Instead of waiting for the full return TD bootstraps from the current estimate of the next states value.

The estimates from MC and TD may differ because they use different learning targets:

MC target:

$$
G_t
$$

TD target:

$$
R_{t+1} + \gamma v(s')
$$

Since $v(s')$ is only an estimate, TD introduces bias.  
MC does not bootstrap but suffers from higher variance because it depends on full trajectory returns.
This is mostly a problem due to the limited data, as both methods will theoretically converge to the same true value function.

#### 2.4 What quantities do you need to have a full learned model $\hat{M}$ of the game?

A full learned model $\hat{M}$ should contain the same quantities as an MDP model:

$$
\hat{M} = \langle S, A, \hat{T}, \hat{R}, \gamma \rangle
$$

where:

- $S$ is the set of states
- $A$ is the set of actions
- $\hat{T}$ is the learned transition function
- $\hat{R}$ is the learned reward function
- $\gamma$ is the discount factor

This follows from the definition of an MDP as

$$
M = \langle S, A, T, R, \gamma \rangle
$$

and from the fact that learning a model means learning the transition and reward functions $\hat{T}$ and $\hat{R}$.

In this situation:

$$
S = \{home, uni\}
$$

$$
A = \{travel, take\ exam\}
$$

So the quantities needed are:

- the states $home$ and $uni$
- the actions $travel$ and $take exam$
- the transition probabilities $\hat{T}(s' \mid s,a)$ for every state-action pair
- the expected rewards $\hat{R}(s,a)$ for every state-action pair
- the chosen discount $\gamma$

By the notation used in the lectures this can be written as 

$$
f_T(s,a;w_T) : S \times A \to S
$$

and

$$
f_R(s,a;w_R) : S \times A \to \mathbb{R}
$$

which represent the learned transition and reward model. 

#### 2.5 Recall the definition of the transition function $T$.

For an MDP, the transition function is the conditional probability of the next state given the current state and action:

$$
T(s' \mid s,a) = P(S_{t+1} = s' \mid S_t = s, A_t = a)
$$

Described in lectures as the transition probability encoding $P(S^{(t+1)} \mid S^{(t)}, A^{(t)})$.

#### 2.6 Infer $T$ from the trajectories using a table.

From the data we observe these state-action-next-state transitions:

- $(home, travel) \to uni$ once
- $(uni, take exam) \to terminal$ eight times

This gives us the table:

| current state $s$ | action $a$ | next state $s'$ | estimated $T(s' \mid s,a)$ |
|---|---|---|---|
| $home$ | $travel$ | $uni$ | $1$ |
| $home$ | $travel$ | $home$ | $0$ |
| $home$ | $travel$ | $terminal$ | $0$ |
| $uni$ | $take exam$ | $terminal$ | $1$ |
| $uni$ | $take exam$ | $uni$ | $0$ |
| $uni$ | $take exam$ | $home$ | $0$ |

From the observed trajectories, the learned transition model is:

$$
\hat{T}(uni \mid home, travel) = 1
$$

$$
\hat{T}(terminal \mid uni, take\ exam) = 1
$$

with the other observed transition probabilities equal to $0$.

#### 2.7 Recall the definition of the reward function $R$.

For an MDP, the reward function maps a state-action pair to a reward:

$$
R : S \times A \to \mathbb{R}
$$

that is,

$$
R(s,a)
$$

returns the immediate reward obtained when taking action $a$ in state $s$.


#### 2.8 Infer $R$ from the trajectories using a table.

From the trajectories we observe rewards for these state-action pairs:

- $(home, travel)$ gives reward $0$ once
- $(uni, take\ exam)$ gives rewards $1,1,0,1,1,1,1,0$

The reward table is:

| state $s$ | action $a$ | observed rewards | estimated $\hat{R}(s,a)$ |
|---|---|---|---|
| $home$ | $travel$ | $0$ | $0$ |
| $uni$ | $take exam$ | $1,1,0,1,1,1,1,0$ | $\frac{3}{4}$ |
| $home$ | $take exam$ | not observed | unknown |
| $uni$ | $travel$ | not observed | unknown |

So we estimate rewards by the empirical mean.

Reward for $(home, travel)$

$$
\hat{R}(home, travel) = 0
$$

Reward for $(uni, take exam)$

$$
\hat{R}(uni, take\ exam) = \frac{1+1+0+1+1+1+1+0}{8} = \frac{6}{8} = \frac{3}{4}
$$



#### 2.9 How does the reward function $R$ differ from the value function vπ(·).

The reward function gives the immediate reward received after taking an action in a state:

$$
R : S \times A \rightarrow \mathbb{R}
$$

For this environment we estimated:

| state $s$ | action $a$ | $\hat{R}(s,a)$ |
|---|---|---|
| $home$ | $travel$ | $0$ |
| $uni$ | $take\ exam$ | $\frac{3}{4}$ |

This means:

- travelling from home gives reward $0$
- taking the exam at uni gives an average reward $\frac{3}{4}$

The reward function therefore describes the immediate outcome of a state–action pair


Value function

The value function instead measures the expected discounted return when following the policy $\pi$ starting from a state

$$
v_\pi(s) = \mathbb{E}_\pi[G_t \mid S_t = s]
$$

From the previous computations we obtained:

| state $s$ | $v_\pi(s)$ |
|---|---|
| $uni$ | $\frac{3}{4}$ |
| $home$ | $\frac{3}{4}\gamma$ |

This value includes future rewards as well and not just the immediate reward.

#### 2.10 Draw the graph for the MDP $\hat{M}$ you have learned.

The learned MDP $\hat{M}$ contains the states

$$
S = \{home,\, uni,\, terminal\}
$$

and the observed transitions:

- $(home, travel) \rightarrow uni$ with reward $0$
- $(uni, take\ exam) \rightarrow terminal$ with reward $\frac{3}{4}$


Graph of the learned MDP

```
(home)
   |
   | travel, r = 0
   v
(uni)
   |
   | take exam, r = 3/4
   v
(terminal)
```



Interpretation:

- From home taking action travel always leads to uni with reward $0$.
- From uni taking action take exam ends the episode and moves to the terminal state
- The expected reward for taking the exam is $\frac{3}{4}$, estimated from the observed trajectories.

The terminal state has no outgoing transitions because the episode ends after taking the exam.

#### 2.11 Using the model $\hat{M}$ that you have inferred, what would be vπ(uni) and vπ(home) computed using DP?

Using the learned model $\hat{M}$, we compute the values with the Bellman expectation equation

$$
v_\pi(s) = \sum_{a \in A} \pi(a \mid s)\left[ R(s,a) + \gamma \sum_{s' \in S} T(s' \mid s,a)\, v_\pi(s') \right]
$$

which is the basis of DP policy evaluation.

From the learned model, we have:

$$
\hat{R}(home, travel) = 0
$$

$$
\hat{R}(uni, take exam) = \frac{3}{4}
$$

and transitions

$$
\hat{T}(uni \mid home, travel) = 1
$$

$$
\hat{T}(terminal \mid uni, take exam) = 1
$$

with

$$
v_\pi(terminal) = 0
$$


Value of $uni$

At $uni$, the policy takes $take exam$, gets expected reward $\frac{3}{4}$, and then the episode terminates.

So

$$
v_\pi(uni) = \frac{3}{4} + \gamma \cdot 0
$$

Hence

$$
v_\pi(uni) = \frac{3}{4}
$$


Value of $home$

At $home$, the policy takes $travel$, gets reward $0$, and moves to $uni$.

So

$$
v_\pi(home) = 0 + \gamma v_\pi(uni)
$$

Substituting $v_\pi(uni) = \frac{3}{4}$ gives

$$
v_\pi(home) = \gamma \cdot \frac{3}{4}
$$


Final answer:

$$
v_\pi(uni) = \frac{3}{4}
$$

$$
v_\pi(home) = \frac{3}{4}\gamma
$$

#### (*) 2.12 How do the state-values you have learned with DP compare to the ones learned by MC and TD? Are they identical or not? If not, can you explain why not?

The state-values learned with DP, MC and TD are not all identical.

Values obtained

| Method | $v_\pi(uni)$ | $v_\pi(home)$ |
|---|---|---|
| MC | $\frac{3}{4}$ | $0$ |
| TD | $\frac{3}{4}$ | $\frac{3}{4}\gamma$ |
| DP | $\frac{3}{4}$ | $\frac{3}{4}\gamma$ |

Thus:

- DP and TD produce the same values.
- MC differs for the state home.


Why MC differs:

Monte-Carlo estimates state values using complete returns from trajectories.

For the state $home$, we only observe one trajectory:

$$
(home, travel, 0), (uni, take exam, 0)
$$

The return from $home$ in that trajectory is

$$
G = 0 + \gamma \cdot 0 = 0
$$

Therefore MC estimates

$$
v_\pi(home) = 0
$$

However, this estimate is based on a single sample and does not capture the expected reward of taking the exam at $uni$.


Why TD and DP match:

TD and DP both rely on the Bellman equation and use the estimated value of the next state.

For $home$:

$$
v_\pi(home) = 0 + \gamma v_\pi(uni)
$$

Since

$$
v_\pi(uni) = \frac{3}{4}
$$

both methods obtain

$$
v_\pi(home) = \frac{3}{4}\gamma
$$

With infinitely many samples the MC estimate would converge to the same value as TD and DP.

#### 2.13 Compute again vπ(uni) and vπ(home) using MC from real and simulated data.

Monte Carlo uses the average of the observed returns.

Value of $uni$:

The state $uni$ appears once in each of the 16 trajectories.

The returns from $uni$ are:

- real data: $1,1,0,1,1,1,1,0$
- simulated data: $1,1,1,1,0,1,1,0$

So the total sum is

$$
6 + 6 = 12
$$

over

$$
8 + 8 = 16
$$

visits.

Therefore

$$
v_\pi(uni) = \frac{12}{16} = \frac{3}{4}
$$

Value of $home$:

The state $home$ appears in:

- real data: $\tau_8$
- simulated data: $\hat{\tau}_3, \hat{\tau}_7, \hat{\tau}_8$

So there are $4$ visits to $home$.

Returns from $home$:

- $\tau_8$: $0 + \gamma \cdot 0 = 0$
- $\hat{\tau}_3$: $0 + \gamma \cdot 1 = \gamma$
- $\hat{\tau}_7$: $0 + \gamma \cdot 1 = \gamma$
- $\hat{\tau}_8$: $0 + \gamma \cdot 0 = 0$

Average:

$$
v_\pi(home) = \frac{0 + \gamma + \gamma + 0}{4} = \frac{2\gamma}{4} = \frac{\gamma}{2}
$$

Results:

$$
v_\pi(uni) = \frac{3}{4}
$$

$$
v_\pi(home) = \frac{\gamma}{2}
$$

#### 2.14 Compute again vπ(uni) and vπ(home) using TD from real and simulated data.

TD uses the Bellman style one step target

$$
R_{t+1} + \gamma v_\pi(s')
$$

and estimates values from one step transitions.

Value of $uni$:

Every transition from $uni$ is:

$$
(uni, take exam, r) \to terminal
$$

So the TD target is just the immediate reward:

$$
r + \gamma v_\pi(terminal) = r
$$

The observed rewards from $uni$ across real and simulated data are again:

$$
12 \text{ ones/zeros summing to } 12 \text{ over } 16 \text{ visits}
$$

so

$$
v_\pi(uni) = \frac{12}{16} = \frac{3}{4}
$$

Value of $home$:

Every transition from $home$ is:

$$
(home, travel, 0) \to uni
$$

This always gives reward $0$ and next state $uni$, so the TD target is

$$
0 + \gamma v_\pi(uni)
$$

Substituting $v_\pi(uni) = \frac{3}{4}$ gives

$$
v_\pi(home) = \gamma \cdot \frac{3}{4}
$$

Results:

$$
v_\pi(uni) = \frac{3}{4}
$$

$$
v_\pi(home) = \frac{3}{4}\gamma
$$

#### (*) 2.15 Did the values computed by MC and TD change? If so, how did they change and why?

The values changed for MC but the TD values stayed the same.

For $uni$:

The value did not change for either MC or TD:

$$
v_\pi(uni) = \frac{3}{4}
$$

This stayed the same because the average exam reward in the simulated data is consistent with the real data.


For $home$:

The value changed for MC:

$$
0 \;\rightarrow\; \frac{\gamma}{2}
$$

but it did not change for TD:

$$
\frac{3}{4}\gamma \;\rightarrow\; \frac{3}{4}\gamma
$$


Why did MC change?

MC estimates values from complete observed returns.

Earlier for $home$, there was only one real trajectory starting from $home$:

$$
(home, travel, 0), (uni, take\ exam, 0)
$$

so the only observed return was

$$
0 + \gamma \cdot 0 = 0
$$

Hence MC estimated

$$
v_\pi(home) = 0
$$

After adding simulated data, we got more trajectories starting from $home$:

- two with return $\gamma$
- two with return $0$

So MC now averages these four returns:

$$
v_\pi(home) = \frac{0 + \gamma + \gamma + 0}{4} = \frac{\gamma}{2}
$$

Thus MC changed because it depends directly on the sample returns and adding more trajectories changes that empirical average.

Why did TD not change?

TD estimates values by bootstrapping from the next state.

For $home$, the transition is always:

$$
(home, travel, 0) \to uni
$$

So TD uses

$$
v_\pi(home) = 0 + \gamma v_\pi(uni)
$$

Since $v_\pi(uni)$ remained

$$
\frac{3}{4}
$$

the TD estimate for $home$ remains

$$
v_\pi(home) = \frac{3}{4}\gamma
$$

So TD did not change because the one step transition structure stayed the same, and the estimate of $uni$ stayed the same.

#### 3. Monte Carlo Tree Search


Considerations to make when deciding on using MCTS: 

- Time constraints
When driving at speed a car needs react to changes in milliseconds, and MCTS relies on simulating a large number of future possibilites. 

- Bad trajectories
MCTS needs to explore alot of trajectories to learn which ones to avoid, that includes bad ones. Exhausting valuable time on simulating undesireable trajectories could have dire consequences many situations. 
The car cannot learn what to do by first doing the wrong thing, that would be like driving into a tree to learn that you shouldnt drive into a tree. Sure you have learnt, but you dont have a car anymore.

- Data completeness
MCTS assumes we have all available data, while in the real world a lot of the surroundings are unknown. Sensors have limited range and fidelity, other drivers behave erratically. The amount of data available to the car is also very large, quickly creating a very large branching factor, especially since alot of the parameters are continuous and not a set of predetermined options.


There are aspects of self-driving that can use MCTS, where time constraints are not relevant, and the available data has simpler parameters. An example could be choosing what lane you should stay in on a highway. It does not require millisecond decision times and it would not use the full array of data avaible to the car, rather high-level data with more predictable inputs. 
